# Imports, configs and Paths

In [29]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

# ---- Step 1: Choose dataset version ----
DATASET_VERSION = 2   # change to 2 to use Data2

DATA_DIR = r"D:/multimodal_pipeline/data"

print("Using dataset:", DATA_DIR)

TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test.tsv")
}

IMAGE_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images")
}

# ---- Task config ----
LABEL_COLUMN = "6_way_label"   # change to "2_way_label" if you want binary
NUM_CLASSES = 6                # set to 2 if you change the label column above

BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3  # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using dataset: D:/multimodal_pipeline/data
Using device: cuda


# Dataset Class (Image-only FakedditDataset)

In [31]:
class FakedditImageDataset(Dataset):
    def __init__(self, tsv_path, images_dir, label_column=LABEL_COLUMN, transform=None):
        self.images_dir = images_dir
        self.label_column = label_column
        self.transform = transform

        df = pd.read_csv(tsv_path, sep="\t")
        print(f"Loaded {len(df)} rows from {tsv_path}")

        # Keep only rows with images if hasImage column exists
        if "hasImage" in df.columns:
            df = df[df["hasImage"] == True]
            print(f"After hasImage filter: {len(df)} rows")

        # Build image_path column
        df["image_path"] = df["id"].astype(str).apply(
            lambda x: os.path.join(images_dir, f"{x}.jpg")
        )

        # Filter to files that actually exist
        df = df[df["image_path"].apply(os.path.exists)]
        print(f"After image file exists filter: {len(df)} rows")

        # Keep only what we need
        self.df = df[["image_path", label_column]].reset_index(drop=True)

        # Check label range
        unique_labels = sorted(self.df[label_column].unique())
        print("Unique labels:", unique_labels)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label = int(row[self.label_column])

        # Load image
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# Transforms & DataLoaders

In [32]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


train_dataset = FakedditImageDataset(
    TSV_FILES["train"], IMAGE_DIRS["train"],
    label_column=LABEL_COLUMN, transform=train_transform
)
val_dataset = FakedditImageDataset(
    TSV_FILES["validate"], IMAGE_DIRS["validate"],
    label_column=LABEL_COLUMN, transform=val_test_transform
)
test_dataset = FakedditImageDataset(
    TSV_FILES["test"], IMAGE_DIRS["test"],
    label_column=LABEL_COLUMN, transform=val_test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # 4 for CPU-only, 8–12 for GPU machines
    pin_memory=False,   # True only if using CUDA
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

len(train_dataset), len(val_dataset), len(test_dataset)


Loaded 564000 rows from D:/multimodal_pipeline/data\multimodal_train.tsv
After hasImage filter: 564000 rows
After image file exists filter: 526629 rows
Unique labels: [0, 1, 2, 3, 4, 5]
Loaded 59342 rows from D:/multimodal_pipeline/data\multimodal_validate.tsv
After hasImage filter: 59342 rows
After image file exists filter: 39343 rows
Unique labels: [0, 1, 2, 3, 4, 5]
Loaded 59319 rows from D:/multimodal_pipeline/data\multimodal_test.tsv
After hasImage filter: 59319 rows
After image file exists filter: 39622 rows
Unique labels: [0, 1, 2, 3, 4, 5]


(526629, 39343, 39622)

# ResNet50 Model for Image Classification

In [33]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models.resnet import Bottleneck

# ===============================================================
# 1. DropPath (Stochastic Depth)
# ===============================================================
class DropPath(nn.Module):
    """
    Drop entire residual paths (Stochastic Depth)
    """
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        # Drop per batch element
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
        return x * mask / keep_prob


# ===============================================================
# 2. Modified Bottleneck with DropPath
# ===============================================================
class BottleneckWithDropPath(Bottleneck):
    """
    Replace the original bottleneck with a version that includes DropPath
    """
    def __init__(self, *args, drop_path_prob=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.drop_path = DropPath(drop_path_prob)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        # APPLY DROPPATH BEFORE ADDING RESIDUAL
        out = self.drop_path(out)

        # Residual connection
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


# ===============================================================
# 3. Function to Inject DropPath Into ResNet
# ===============================================================
def apply_stochastic_depth(model, drop_prob=0.1):
    """
    Replace all Bottleneck blocks inside ResNet-50 with DropPath versions
    """
    for name, module in model.named_modules():
        if isinstance(module, Bottleneck):
            # Replace with our modified version
            module.drop_path = DropPath(drop_prob)


# ===============================================================
# 4. Image-Only ResNet50 (with DropPath + classifier head)
# ===============================================================
class ImageOnlyResNet50(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=False,
                 dropout=0.5, drop_path_prob=0.1):
        super().__init__()

        # Base ResNet-50
        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        )

        # Inject DropPath into all bottleneck blocks
        apply_stochastic_depth(self.backbone, drop_prob=drop_path_prob)

        # Optionally freeze backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Replace the final classification head
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


# Training Utilities (Early Stopping + Curves)

In [34]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state_dict = None
        self.should_stop = False

    def step(self, loss, model):
        if loss + self.min_delta < self.best_loss:
            self.best_loss = loss
            self.counter = 0
            self.best_state_dict = model.state_dict()
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True



def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_acc += (preds == labels).sum().item()

    return total_loss/len(loader.dataset), total_acc/len(loader.dataset)


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * images.size(0)
            total_acc += (preds == labels).sum().item()

    return total_loss/len(loader.dataset), total_acc/len(loader.dataset)



def plot_learning_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curves")
    plt.legend()
    plt.grid(True)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curves")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


# Full Training Loop

In [35]:
pip install --upgrade torch torchvision torchaudio


Note: you may need to restart the kernel to use updated packages.


In [36]:
batch = next(iter(train_loader))
print("Batch loaded!")
for b in batch:
    print(type(b), b.shape if hasattr(b, 'shape') else None)


Batch loaded!
<class 'torch.Tensor'> torch.Size([32, 3, 224, 224])
<class 'torch.Tensor'> torch.Size([32])


In [37]:
for i in range(5):
    x = train_dataset[i]
    print("Loaded sample:", i)


Loaded sample: 0
Loaded sample: 1
Loaded sample: 2
Loaded sample: 3
Loaded sample: 4


In [38]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True
NVIDIA GeForce RTX 4060 Ti


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ImageOnlyResNet50(
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_backbone=False,
    dropout=0.5,
    drop_path_prob=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2)

early = EarlyStopping(patience=3)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"Val   Loss={val_loss:.4f}, Acc={val_acc:.4f}")

    early.step(val_loss, model)
    if early.should_stop:
        print("Early stopping activated.")
        break

# restore best model
if early.best_state_dict:
    model.load_state_dict(early.best_state_dict)

torch.save(model.state_dict(), "best_image_model_resnet50_droppath.pth")
print("Model saved!")


NameError: name 'EPOCHS' is not defined

# Evaluation on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader):
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))


Classification report (test):
              precision    recall  f1-score   support

           0     0.7608    0.9124    0.8297     22002
           1     0.7963    0.6095    0.6905      3291
           2     0.7369    0.5737    0.6451     10518
           3     0.3185    0.1019    0.1544      1197
           4     0.9024    0.7592    0.8246       353
           5     0.8023    0.6568    0.7223      2261

    accuracy                         0.7569     39622
   macro avg     0.7195    0.6023    0.6445     39622
weighted avg     0.7477    0.7569    0.7426     39622

Confusion matrix:
 [[20074   197  1498    49    16   168]
 [ 1003  2006   170    52     3    57]
 [ 4063   184  6034   114     8   115]
 [  677    32   340   122     2    24]
 [   47    11    25     0   268     2]
 [  520    89   121    46     0  1485]]
